# 04 - Landmark-only 15-class baseline

Train the baseline PyTorch MLP on the 15 configured classes using cached MediaPipe landmarks, presence masks, and handedness-reliability masks. Run notebooks 02 and 03 first. Notebook 08 compares this model with the improved model.

The epoch is selected on validation loss, then the baseline is refit on train plus validation signers for that fixed epoch count. The test split is reserved for notebook 08.

In [ ]:
import copy
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, project_path
from src.dataset import load_split_manifest, split_fingerprint, validate_split_groups
from src.landmark_extractor import HAND_FEATURE_DIM, LANDMARK_CACHE_FORMAT_VERSION, LANDMARK_FEATURE_DIM
from src.metrics import save_evaluation
from src.utils import plot_history, set_seed

config = load_config()
seed = int(config['project'].get('seed', 42))
set_seed(seed)
torch.set_num_threads(min(torch.get_num_threads(), 4))
manifest_path = project_path(config, 'splits') / 'split_manifest.json'
cache_path = project_path(config, 'landmarks') / 'mediapipe_landmarks.json'
manifest = load_split_manifest(manifest_path)
if not cache_path.is_file():
    raise FileNotFoundError(f'Landmark cache not found: {cache_path}. Run notebook 03 first.')
cache = json.loads(cache_path.read_text(encoding='utf-8'))
if cache.get('format_version') != LANDMARK_CACHE_FORMAT_VERSION or cache.get('feature_dim') != LANDMARK_FEATURE_DIM or cache.get('input_is_mirrored', False) != bool(config['data'].get('images_are_mirrored', False)):
    raise ValueError('Landmark cache format or mirrored-input setting does not match the configuration; rerun notebook 03.')
class_to_idx = manifest['class_to_idx']
class_names = [name for name, _ in sorted(class_to_idx.items(), key=lambda pair: pair[1])]
if sorted(class_to_idx.values()) != list(range(len(class_names))):
    raise ValueError('Class indices must be consecutive integers starting at zero.')
rows_by_split = manifest['splits']
all_rows = [row for rows in rows_by_split.values() for row in rows]
keys = [row['relative_path'] for row in all_rows]
if len(keys) != len(set(keys)) or set(keys) != set(cache['items']):
    raise ValueError('Split manifest and landmark cache have duplicate, missing, or extra image keys; rerun notebooks 02 and 03.')
validate_split_groups(manifest)

arrays = {}
for name, rows in rows_by_split.items():
    features, labels = [], []
    for row in rows:
        item = cache['items'][row['relative_path']]
        if any(item.get(key) != row[key] for key in ('sha256', 'label', 'label_idx')) or class_to_idx.get(row['label']) != row['label_idx']:
            raise ValueError(f"Stale or misaligned landmark data: {row['relative_path']}")
        if len(item['features']) != LANDMARK_FEATURE_DIM or len(item['positions']) != 8 or len(item['presence']) != 2 or len(item['handedness_reliable']) != 2:
            raise ValueError(f"Invalid landmark feature shape: {row['relative_path']}")
        features.append(item['features'][:HAND_FEATURE_DIM] + item['presence'] + item['handedness_reliable'])
        labels.append(row['label_idx'])
    x = np.asarray(features, dtype=np.float32)
    y = np.asarray(labels, dtype=np.int64)
    if x.shape != (len(rows), HAND_FEATURE_DIM + 4) or not np.isfinite(x).all() or set(y) != set(range(len(class_names))):
        raise ValueError(f'Invalid feature values or incomplete class coverage in {name} split.')
    arrays[name] = (torch.from_numpy(x), torch.from_numpy(y))
    no_hand = int((x[:, HAND_FEATURE_DIM:HAND_FEATURE_DIM + 2].sum(axis=1) == 0).sum())
    print(f'{name}: {len(rows)} images; no hands detected: {no_hand} ({no_hand / len(rows):.1%})')
print('Split strategy:', manifest.get('split_mode'))
print('Signer IDs:', {name: sorted({row['signer_id'] for row in rows}) for name, rows in rows_by_split.items()})

## Train, select on validation, and refit on all non-test signers

Select the epoch using validation loss, then retrain the same MLP for that fixed number of epochs on combined train and validation data. The held-out test signer is not used in training or selection.

In [ ]:
model = nn.Sequential(
    nn.Linear(HAND_FEATURE_DIM + 4, 256), nn.ReLU(), nn.Dropout(0.25),
    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.25),
    nn.Linear(128, len(class_names)),
)
train_loader = DataLoader(TensorDataset(*arrays['train']), batch_size=256, shuffle=True, generator=torch.Generator().manual_seed(seed))
validation_x, validation_y = arrays['validation']
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
criterion = nn.CrossEntropyLoss()
max_epochs, patience = 100, 12
best_loss, best_epoch, stale_epochs, best_state = float('inf'), 0, 0, None
history = []
started = time.perf_counter()
for epoch in range(1, max_epochs + 1):
    model.train()
    total_loss, total_correct = 0.0, 0
    for features, labels in train_loader:
        optimizer.zero_grad(set_to_none=True)
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
    model.eval()
    with torch.inference_mode():
        validation_logits = model(validation_x)
        validation_loss = criterion(validation_logits, validation_y).item()
        validation_accuracy = (validation_logits.argmax(dim=1) == validation_y).float().mean().item()
    row = {'epoch': epoch, 'train_loss': total_loss / len(train_loader.dataset),
           'train_accuracy': total_correct / len(train_loader.dataset),
           'validation_loss': validation_loss, 'validation_accuracy': validation_accuracy}
    history.append(row)
    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:3d}: train loss {row['train_loss']:.4f}, validation loss {validation_loss:.4f}, validation accuracy {validation_accuracy:.2%}")
    if validation_loss < best_loss:
        best_loss, best_epoch, stale_epochs = validation_loss, epoch, 0
        best_state = copy.deepcopy(model.state_dict())
    else:
        stale_epochs += 1
        if stale_epochs >= patience:
            print(f'Early stopping after epoch {epoch}.')
            break
training_seconds = time.perf_counter() - started
if best_state is None:
    raise RuntimeError('No valid validation checkpoint was produced.')
selected_epoch = best_epoch
combined_x = torch.cat((arrays['train'][0], arrays['validation'][0]))
combined_y = torch.cat((arrays['train'][1], arrays['validation'][1]))
set_seed(seed)
model = nn.Sequential(
    nn.Linear(HAND_FEATURE_DIM + 4, 256), nn.ReLU(), nn.Dropout(0.25),
    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.25),
    nn.Linear(128, len(class_names)),
)
refit_loader = DataLoader(
    TensorDataset(combined_x, combined_y), batch_size=256, shuffle=True,
    generator=torch.Generator().manual_seed(seed),
)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
for _ in range(selected_epoch):
    model.train()
    for features, labels in refit_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(features), labels)
        loss.backward()
        optimizer.step()
model.eval()
training_seconds = time.perf_counter() - started
checkpoint_dir = project_path(config, 'checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / 'landmark_mlp_best.pt'
validation_checkpoint_path = checkpoint_dir / 'landmark_mlp_validation_best.pt'
torch.save({
    'model_type': 'landmark_mlp', 'model_state_dict': best_state, 'class_to_idx': class_to_idx,
    'input_dim': HAND_FEATURE_DIM + 4, 'hidden_dims': [256, 128], 'dropout': 0.25,
    'epoch': selected_epoch, 'validation_loss': best_loss,
    'validation_accuracy': history[selected_epoch - 1]['validation_accuracy'], 'seed': seed,
    'split_fingerprint': split_fingerprint(manifest),
    'input_is_mirrored': cache.get('input_is_mirrored', False),
}, validation_checkpoint_path)
torch.save({'model_type': 'landmark_mlp', 'model_state_dict': model.state_dict(), 'class_to_idx': class_to_idx,
            'input_dim': HAND_FEATURE_DIM + 4, 'hidden_dims': [256, 128], 'dropout': 0.25,
            'epoch': selected_epoch, 'validation_loss': best_loss, 'validation_accuracy': history[selected_epoch - 1]['validation_accuracy'], 'seed': seed, 'split_fingerprint': split_fingerprint(manifest),
            'training_seconds': training_seconds, 'input_is_mirrored': cache.get('input_is_mirrored', False)}, checkpoint_path)
pd.DataFrame(history).to_csv(checkpoint_dir / 'landmark_mlp_history.csv', index=False)
plot_path = plot_history(history, project_path(config, 'plots') / 'landmark_mlp_curves.png', 'Landmark MLP')
print(f'Validation-selected epoch: {selected_epoch}; validation loss: {best_loss:.4f}; total training time: {training_seconds:.1f} seconds')
print(f'Saved final refit checkpoint: {checkpoint_path}')
print(f'Saved validation-only checkpoint for model selection: {validation_checkpoint_path}')